In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from fastai.vision.all import load_learner


In [3]:
DS = Path("../../datasets")
POS = DS / "positive-samples"

OUT_MAIN = DS / "test-pos-samples"
OUT_MAIN.mkdir(parents=True, exist_ok=True)
IMAGES = list(POS.glob("*.jpg"))
IMAGES[:2]

OUT_MAIN

Path('../../datasets/test-pos-samples')

In [4]:
# MODEL_CODE = "scale=100_to_200;ds=3000_to_200"
# MODEL_PATH = "/Users/hariomnarang/Desktop/gdrive-sync/garbage/experiments/T009-engulf-3000-200/log/export_iter_4.pkl"
# TILE_SIZE = 200
# SHRINK_TO = 200

VERSION = "stridded_mapillary_neg"

EXPERIMENTS = [
    # {
    #     "model_code": "scale=100_to_200;ds=3000_to_200",
    #     "model_path": "/Users/hariomnarang/Desktop/gdrive-sync/garbage/experiments/T009-engulf-3000-200/log/export_iter_4.pkl",
    #     "tile_size": 100,
    #     "shrink_to": 200,
    # },
    # {
    #     "model_code": "scale=200_to_200;ds=3000_to_200",
    #     "model_path": "/Users/hariomnarang/Desktop/gdrive-sync/garbage/experiments/T009-engulf-3000-200/log/export_iter_4.pkl",
    #     "tile_size": 200,
    #     "shrink_to": 200,
    # },
    # {
    #     "model_code": "scale=100_to_50",
    #     "model_path": "/Users/hariomnarang/Desktop/gdrive-sync/garbage/experiments/T008-engulf-v2-only-uncentered/log/export_iter_4.pkl",
    #     "tile_size": 100,
    #     "shrink_to": 50,
    # },
    {
        "model_code": "scale=50_to_50",
        "model_path": "/Users/hariomnarang/Desktop/gdrive-sync/garbage/experiments/T008-engulf-v2-only-uncentered/log/export_iter_4.pkl",
        "tile_size": 50,
        "shrink_to": 50,
    },
]

OUT = OUT_MAIN / VERSION
OUT.mkdir(parents=True, exist_ok=True)

In [5]:
from mtrain.smallnet.tfms import resize_and_pad_raw, unpad_and_resize_mask
from PIL import Image
from itertools import batched, chain
from mtrain.seg.cityscapes import cached_predict, get_mask_with_labels, CityScapesCls
from tqdm import tqdm
from mtrain.smallnet.tile import split_image_into_tiles
from mtrain.seg import mapillary as mapi


def red(string):
    return "\x1b[31m" + string + "\x1b[0m"


def draw_border_inplace(out, thickness=1, value=255):
    h, w = out.shape[:2]

    out[:thickness, :] = value
    out[h - thickness : h, :] = value
    out[:, :thickness] = value
    out[:, w - thickness : w] = value


def predict_unet(img_path, sz, learner, alpha=0.4):
    img_arr = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    arr_and_coord = split_image_into_tiles(img_arr, sz)
    mask_and_coord = [
        (learner.predict(arr)[0].numpy(), (y, x))
        for (arr, (y, x)) in tqdm(arr_and_coord)
    ]
    res = img_arr.copy()
    H, W = res.shape[:2]
    for mask, (y, x) in tqdm(mask_and_coord):
        ny, nx = min(y + sz, H), min(x + sz, W)
        roi = res[y:ny, x:nx]
        mask = mask[: ny - y, : nx - x]
        mask = mask.astype(bool)

        roi[mask] = (
            (1 - alpha) * roi[mask].astype(np.float32) + alpha * np.array([255, 0, 0])
        ).astype(np.uint8)
    return res


def _with_mapillary_negmask(img_path, orig_mask):
    pred = mapi.cached_predict(img_path)
    neg_mask = mapi.get_mask_with_labels(
        pred,
        [
            mapi.Label.ROAD,
            mapi.Label.VEGETATION,
            mapi.Label.BIKE_LANE,
            mapi.Label.SIDEWALK,
            mapi.Label.SAND,
        ],
    )
    return orig_mask & neg_mask


def _with_city_scapes_negmask(img_path, orig_mask):
    pred = cached_predict(img_path)
    neg_mask = get_mask_with_labels(
        pred,
        [
            CityScapesCls.ROAD,
            CityScapesCls.SIDEWALK,
            CityScapesCls.TERRAIN,
            CityScapesCls.VEGETATION,
        ],
    )
    return orig_mask & neg_mask


def _do_batch(batch, learner):
    images = [Image.fromarray(arr) for (arr, _, _) in batch]

    tdl = learner.dls.test_dl(test_items=images, with_labels=False)
    preds, _ = learner.get_preds(dl=tdl)
    masks = preds.argmax(dim=1)

    return [(mask.numpy(), meta, pos) for mask, (_, meta, pos) in zip(masks, batch)]


def _run_predict(mask_and_coord, learner, bs=8):
    batched_results = (
        _do_batch(batch, learner) for batch in tqdm(batched(mask_and_coord, bs))
    )
    return list(chain.from_iterable(batched_results))


def _predict_unet_with_resizer_img_arr(
    img_arr_rgb, tile_size, after_resize_size, learner
):
    img_arr = img_arr_rgb
    arr_and_coord = split_image_into_tiles(img_arr, tile_size)

    mask_and_coord = []
    for arr, pos in arr_and_coord:
        resized, meta = resize_and_pad_raw(arr, after_resize_size)
        mask_and_coord.append((resized, meta, pos))

    mask_and_coord = _run_predict(mask_and_coord, learner)

    mask_and_coord = [
        (unpad_and_resize_mask(arr, meta), pos) for (arr, meta, pos) in mask_and_coord
    ]

    H, W = img_arr.shape[:2]
    full_mask = np.zeros((H, W)).astype(bool)
    for mask, (y, x) in tqdm(mask_and_coord):
        ny, nx = min(y + tile_size, H), min(x + tile_size, W)
        mask = mask[: ny - y, : nx - x]
        mask = mask.astype(bool)
        full_mask[y:ny, x:nx] = mask

    return full_mask


def _shift_mask(mask, orig_h, orig_w, shift_y, shift_x):
    H, W = orig_h, orig_w
    aligned = np.zeros((H, W), dtype=mask.dtype)
    aligned[shift_y:, shift_x:] = mask
    return aligned


def predict_unet_with_resizer(
    img_path, tile_size, after_resize_size, learner, alpha=0.4
):
    img_arr = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    strided = img_arr[25:, 25:]
    fm1 = _predict_unet_with_resizer_img_arr(
        img_arr, tile_size, after_resize_size, learner
    )
    fm2 = _predict_unet_with_resizer_img_arr(
        strided, tile_size, after_resize_size, learner
    )
    H, W = fm1.shape[:2]
    fm2 = _shift_mask(fm2, H, W, 25, 25)
    full_mask = fm1 | fm2
    full_mask = _with_mapillary_negmask(img_path, full_mask)
    res = img_arr.copy()
    res[full_mask] = (
        (1 - alpha) * res[full_mask].astype(np.float32) + alpha * np.array([255, 0, 0])
    ).astype(np.uint8)
    return res, full_mask

In [6]:
def run_single_exp(model_path, model_code, tile_size, shrink_to):
    print(red(f"running: {model_code}"))
    learner = load_learner(model_path)
    outd = OUT / model_code
    outd.mkdir(parents=True, exist_ok=True)

    for i, img in enumerate(IMAGES):
        print(red(f"{i} / {len(IMAGES)}"))
        r, _ = predict_unet_with_resizer(img, tile_size, shrink_to, learner)
        plt.imsave(outd / f"{i}.jpg", r)

In [7]:
for exp in EXPERIMENTS:
    run_single_exp(**exp)

/Users/hariomnarang/Desktop/personal/roads/mtrain/.venv/lib/python3.12/site-packages/fastai/learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


running: scale=50_to_50
0 / 80


32it [00:05,  6.15it/s]
100%|██████████| 252/252 [00:00<00:00, 133759.13it/s]
30it [00:05,  5.30it/s]
100%|██████████| 240/240 [00:00<00:00, 96236.42it/s]


1 / 80


32it [00:05,  5.55it/s]
100%|██████████| 252/252 [00:00<00:00, 154889.30it/s]
30it [00:04,  6.28it/s]
100%|██████████| 240/240 [00:00<00:00, 183893.49it/s]


...

42it [00:05,  7.19it/s]
100%|██████████| 336/336 [00:00<00:00, 272747.46it/s]
38it [00:05,  7.11it/s]
100%|██████████| 300/300 [00:00<00:00, 279558.14it/s]


78 / 80


32it [00:04,  7.48it/s]
100%|██████████| 252/252 [00:00<00:00, 246321.28it/s]
30it [00:03,  7.61it/s]
100%|██████████| 240/240 [00:00<00:00, 198821.44it/s]


79 / 80


35it [00:04,  7.52it/s]
100%|██████████| 273/273 [00:00<00:00, 243057.74it/s]
33it [00:04,  7.43it/s]
100%|██████████| 260/260 [00:00<00:00, 257197.89it/s]


In [35]:
photo_to_code = OUT / "photo-to-code"
photo_to_code.mkdir(parents=True, exist_ok=True)
photo_to_code = photo_to_code.resolve()

for img in tqdm(OUT.rglob("*.jpg")):
    img = img.resolve()
    if img.is_relative_to(photo_to_code):
        continue

    img_dir = photo_to_code / img.stem
    img_dir.mkdir(parents=True, exist_ok=True)
    link = img_dir / f"{img.parent.name}.jpg"
    if link.exists():
        link.unlink()
    link.symlink_to(img)

640it [00:00, 2812.12it/s]
